# Wreckognise — GPU training on Kaggle

Same pipeline as the Colab notebook, for Kaggle's GPU — **30 hours a week free**, far more
generous than Colab's free tier.

**Bar to beat:** the CPU run scored `SCTD 0.774` / `cross-dataset 0.349`.

---

## Set-up, once

**1. GPU on** — right panel → *Session options* → **Accelerator: GPU T4 x2**.
   (Needs a phone-verified account: Settings → Phone Verification.)

**2. Internet on** — same panel → **Internet: On**. Required to fetch SCTD.

**3. Attach AI4Shipwrecks** — right panel → **Add Input** → search `ai4shipwrecks`
   → add **`doanduchieu/ai4shipwrecks`** (public, MIT). No upload needed.

   If you would rather use your own copy, upload `AI4Shipwrecks.zip` as a New Dataset and
   attach that instead — the notebook finds the data by structure, not by name, so either works.

Cells are idempotent and training resumes from its last checkpoint, so re-running after a
session restart costs minutes rather than the whole run.


## 1. Confirm the GPU is attached


In [ ]:
import torch, subprocess

print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
      or 'no nvidia-smi — you are on CPU')
print('torch sees CUDA:', torch.cuda.is_available())

assert torch.cuda.is_available(), (
    'No GPU. Right panel -> Session options -> Accelerator: GPU T4 x2.'
)


## 2. Install


In [ ]:
!pip -q install ultralytics onnx onnxslim
import ultralytics; print('ultralytics', ultralytics.__version__)


## 3. Locate inputs and working directory

Kaggle mounts attached datasets read-only under `/kaggle/input/`. Everything written goes to
`/kaggle/working/`, which persists across session restarts and is what you download at the end.


In [ ]:
from pathlib import Path

WORK = Path('/kaggle/working')          # persists across restarts; downloadable
WORK.mkdir(parents=True, exist_ok=True)


def find_ai4shipwrecks(root=Path('/kaggle/input')):
    """Locate the dataset by SHAPE rather than by name.

    Public Kaggle mirrors and a hand-uploaded zip lay the files out differently,
    and dataset slugs change. What does not change is the structure: a directory
    holding train/images alongside train/labels. Search for that.
    """
    if not root.exists():
        return None, None
    for cand in root.rglob('train'):
        if (cand/'images').is_dir() and (cand/'labels').is_dir():
            return cand.parent, None            # already extracted
    for cand in root.rglob('*.zip'):
        if 'ai4' in cand.name.lower() or 'shipwreck' in cand.name.lower():
            return None, cand                   # needs extracting
    return None, None


PREEXTRACTED, ZIP = find_ai4shipwrecks()

attached = [p.name for p in Path('/kaggle/input').glob('*')] if Path('/kaggle/input').exists() else []
print('inputs attached:', attached or 'NONE')
print('pre-extracted  :', PREEXTRACTED)
print('zip found      :', ZIP)
if PREEXTRACTED is None and ZIP is None:
    print()
    print('  -> Add Input -> search ai4shipwrecks -> add doanduchieu/ai4shipwrecks, then re-run.')
    print('     Training still works on SCTD alone, but transfer will be poor.')


## 4. Fetch the datasets


In [ ]:
import os, subprocess, zipfile
from pathlib import Path

# --- SCTD: needs Internet: On ---
if not Path('/kaggle/working/sctd').exists():
    subprocess.run(['git','clone','-q','--depth','1',
                    'https://github.com/MingqiangNing/SCTD.git',
                    '/kaggle/working/sctd_repo'], check=True)
    with zipfile.ZipFile('/kaggle/working/sctd_repo/SCTD.zip') as z:
        z.extractall('/kaggle/working/sctd')
print('SCTD images:', len(list(Path('/kaggle/working/sctd').rglob('*.jpg'))),
      '| annotations:', len(list(Path('/kaggle/working/sctd').rglob('*.xml'))))

# --- AI4Shipwrecks ---
AI4 = Path('/kaggle/working/ai4sw/AI4Shipwrecks')
if AI4.exists():
    print('AI4Shipwrecks already available')
elif PREEXTRACTED is not None:
    # /kaggle/input is read-only but readable in place, so symlink rather
    # than burn several minutes copying 2.4 GB.
    AI4.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(PREEXTRACTED, AI4)
    print('using pre-extracted dataset at', PREEXTRACTED)
elif ZIP is not None:
    print('extracting 1.2 GB, ~1-2 min...')
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/kaggle/working/ai4sw')
else:
    print('AI4Shipwrecks not attached — continuing with SCTD only')

if AI4.exists():
    print('  train:', len(os.listdir(AI4/'train'/'images')),
          '| test:', len(os.listdir(AI4/'test'/'images')))


## 5. Build the combined YOLO dataset

Two decisions that matter here:

* **AI4Shipwrecks is tiled to 512 px, not downscaled.** Its images are 5579×1728 full swaths
  where a wreck is ~2.5% of the frame; squashed whole to 640 px a wreck becomes ~15 px and is
  unlearnable. Tiling also matches how the API runs inference.
* **Empty-seabed tiles are kept** as negatives. They are what teaches the model not to fire on
  ripple texture — the main source of false positives.

Splits follow each dataset's own boundary, so no tile of a wreck leaks between train and val.


In [ ]:
import random, shutil, xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path
import cv2, numpy as np

CLASSES = ['aircraft', 'human', 'ship']
SHIP_ID = CLASSES.index('ship')
OUT = Path('/kaggle/working/dataset')
TILE, STRIDE = 512, 384
MIN_BLOB_PX, MIN_BOX_PX, NEG_RATIO = 60, 12, 0.6
rng = random.Random(1337)

if (OUT/'data.yaml').is_file():
    print('dataset already built — skipping (delete /kaggle/working/dataset to rebuild)')
else:
    for s in ('train','val'):
        (OUT/'images'/s).mkdir(parents=True, exist_ok=True)
        (OUT/'labels'/s).mkdir(parents=True, exist_ok=True)

    # ---------- SCTD: Pascal VOC -> YOLO, stratified 80/20 ----------
    def parse_voc(x):
        try: root = ET.parse(x).getroot()
        except ET.ParseError: return None, None, []
        sz = root.find('size')
        if sz is None: return None, None, []
        w, h = int(float(sz.findtext('width'))), int(float(sz.findtext('height')))
        out = []
        for o in root.findall('object'):
            n = (o.findtext('name') or '').strip().lower()
            bb = o.find('bndbox')
            if not n or bb is None or n not in CLASSES: continue
            x0,y0 = float(bb.findtext('xmin')), float(bb.findtext('ymin'))
            x1,y1 = float(bb.findtext('xmax')), float(bb.findtext('ymax'))
            x0,x1 = sorted((max(0,x0), min(w,x1))); y0,y1 = sorted((max(0,y0), min(h,y1)))
            if x1-x0 >= 2 and y1-y0 >= 2: out.append((n,x0,y0,x1,y1))
        return w, h, out

    imgs = {p.stem: p for p in Path('/kaggle/working/sctd').rglob('*.jpg')}
    xmls = {p.stem: p for p in Path('/kaggle/working/sctd').rglob('*.xml')}
    samples = []
    for k in sorted(imgs.keys() & xmls.keys()):
        w,h,b = parse_voc(xmls[k])
        if w and h and b: samples.append((k,w,h,b))

    by_cls = defaultdict(list)
    for e in samples:
        by_cls[Counter(x[0] for x in e[3]).most_common(1)[0][0]].append(e)
    train, val = [], []
    for _, entries in by_cls.items():
        rng.shuffle(entries)
        cut = max(1, round(len(entries)*0.2))
        val += entries[:cut]; train += entries[cut:]

    for split, entries in (('train',train), ('val',val)):
        for k,w,h,boxes in entries:
            shutil.copy2(imgs[k], OUT/'images'/split/f'{k}.jpg')
            lines = [f'{CLASSES.index(n)} {((x0+x1)/2)/w:.6f} {((y0+y1)/2)/h:.6f}'
                     f' {(x1-x0)/w:.6f} {(y1-y0)/h:.6f}'
                     for n,x0,y0,x1,y1 in boxes]
            (OUT/'labels'/split/f'{k}.txt').write_text('\n'.join(lines))
    print(f'SCTD -> train {len(train)}, val {len(val)}')

    # ---------- AI4Shipwrecks: masks -> tiled boxes ----------
    ai4 = Path('/kaggle/working/ai4sw/AI4Shipwrecks')
    if ai4.exists():
        def mask_boxes(m):
            n,_,st,_ = cv2.connectedComponentsWithStats((m>0).astype(np.uint8), 8)
            return [tuple(int(v) for v in st[i][:4]) for i in range(1,n)
                    if st[i][4] >= MIN_BLOB_PX and st[i][2] >= 4 and st[i][3] >= 4]

        for src, dstsplit in (('train','train'), ('test','val')):
            pos, neg = [], []
            for ip in sorted((ai4/src/'images').glob('*.png')):
                im = cv2.imread(str(ip), cv2.IMREAD_GRAYSCALE)
                mk = cv2.imread(str(ai4/src/'labels'/ip.name), cv2.IMREAD_GRAYSCALE)
                if im is None or mk is None: continue
                if mk.shape != im.shape:
                    mk = cv2.resize(mk, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST)
                boxes = mask_boxes(mk); H, W = im.shape
                for top in range(0, max(1,H-TILE+1), STRIDE):
                    for left in range(0, max(1,W-TILE+1), STRIDE):
                        bot, right = min(top+TILE,H), min(left+TILE,W)
                        if bot-top < TILE//2 or right-left < TILE//2: continue
                        local = []
                        for bx,by,bw,bh in boxes:
                            ix0,iy0 = max(bx,left), max(by,top)
                            ix1,iy1 = min(bx+bw,right), min(by+bh,bot)
                            if ix1-ix0 < MIN_BOX_PX or iy1-iy0 < MIN_BOX_PX: continue
                            if (ix1-ix0)*(iy1-iy0) < 0.35*bw*bh: continue
                            local.append((ix0-left, iy0-top, ix1-ix0, iy1-iy0))
                        rec = (ip, left, top, right-left, bot-top, local)
                        (pos if local else neg).append(rec)
            rng.shuffle(neg); neg = neg[:int(len(pos)*NEG_RATIO)]
            for i,(ip,left,top,tw,th,boxes) in enumerate(pos+neg):
                im = cv2.imread(str(ip), cv2.IMREAD_GRAYSCALE)
                stem = f'ai4_{ip.stem}_{left}_{top}_{i}'
                cv2.imwrite(str(OUT/'images'/dstsplit/f'{stem}.jpg'), im[top:top+th, left:left+tw])
                (OUT/'labels'/dstsplit/f'{stem}.txt').write_text('\n'.join(
                    f'{SHIP_ID} {(bx+bw/2)/tw:.6f} {(by+bh/2)/th:.6f} {bw/tw:.6f} {bh/th:.6f}'
                    for bx,by,bw,bh in boxes))
            print(f'AI4SW {src} -> {dstsplit}: {len(pos)} positive + {len(neg)} empty tiles')

    (OUT/'data.yaml').write_text(
        f'path: {OUT}\ntrain: images/train\nval: images/val\n\nnames:\n'
        + ''.join(f'  {i}: {c}\n' for i,c in enumerate(CLASSES)))

print('TOTAL train:', len(list((OUT/'images'/'train').glob('*.jpg'))),
      '| val:', len(list((OUT/'images'/'val').glob('*.jpg'))))


## 6. Train

`yolov8s` at 640 px — bigger model and higher resolution than the CPU run could afford.

Weights are written straight to Drive, and the cell **auto-resumes** from the last checkpoint
if a disconnect interrupts it. Re-running after a drop costs you minutes, not the whole run.

`flipud=0.0` is deliberate: a vertical flip puts the acoustic shadow on the wrong side of the
target, destroying the strongest cue the model has. Horizontal flip is fine — it swaps port
and starboard, which is physically valid.


In [ ]:
from ultralytics import YOLO
from pathlib import Path

RUNS = str(WORK)              # persists across session restarts
NAME = 'run'
LAST = Path(RUNS)/NAME/'weights'/'last.pt'

resume = LAST.is_file()
print('resuming from checkpoint' if resume else 'starting fresh')

model = YOLO(str(LAST)) if resume else YOLO('yolov8s.pt')
model.train(
    data='/kaggle/working/dataset/data.yaml',
    epochs=150, imgsz=640, batch=16, device=0,
    project=RUNS, name=NAME, exist_ok=True, resume=resume,
    patience=40, seed=1337, deterministic=True, plots=True,
    save_period=10,        # checkpoint to Drive every 10 epochs
    # --- sonar-aware augmentation ---
    fliplr=0.5,            # port/starboard swap: physically valid
    flipud=0.0,            # would invert shadow direction: never
    degrees=5.0, translate=0.10, scale=0.5, shear=0.0, perspective=0.0,
    mosaic=1.0, close_mosaic=15, mixup=0.1,
    hsv_h=0.0, hsv_s=0.0,  # sonar is single-channel intensity
    hsv_v=0.4,             # gain variation between surveys is real
)


## 7. Measure it

Scores the best checkpoint on the combined split **and** on AI4Shipwrecks alone. The second
number is the one that says whether the model generalises rather than memorising one corpus.

**Beat 0.349 and it is worth deploying.**


In [ ]:
import shutil
from pathlib import Path
from ultralytics import YOLO

BEST = str(Path(RUNS)/NAME/'weights'/'best.pt')
res = {}

b = YOLO(BEST).val(data='/kaggle/working/dataset/data.yaml', imgsz=640, device=0, plots=False).box
res['combined'] = dict(map50=float(b.map50), map=float(b.map), p=float(b.mp), r=float(b.mr))

# AI4Shipwrecks-only split, for the cross-dataset read
sub = Path('/kaggle/working/ai4_only')
if sub.exists(): shutil.rmtree(sub)
for s in ('train','val'):
    (sub/'images'/s).mkdir(parents=True, exist_ok=True)
    (sub/'labels'/s).mkdir(parents=True, exist_ok=True)

tiles = sorted(Path('/kaggle/working/dataset/images/val').glob('ai4_*.jpg'))
for p in tiles:
    shutil.copy2(p, sub/'images'/'val'/p.name)
    lp = Path('/kaggle/working/dataset/labels/val')/f'{p.stem}.txt'
    if lp.is_file(): shutil.copy2(lp, sub/'labels'/'val'/lp.name)
for p in tiles[:2]:
    shutil.copy2(p, sub/'images'/'train'/p.name)
    shutil.copy2(sub/'labels'/'val'/f'{p.stem}.txt', sub/'labels'/'train'/f'{p.stem}.txt')
(sub/'data.yaml').write_text(
    f'path: {sub}\ntrain: images/train\nval: images/val\n\n'
    'names:\n  0: aircraft\n  1: human\n  2: ship\n')

if tiles:
    b2 = YOLO(BEST).val(data=str(sub/'data.yaml'), imgsz=640, device=0, plots=False).box
    res['ai4shipwrecks_only'] = dict(map50=float(b2.map50), map=float(b2.map),
                                     p=float(b2.mp), r=float(b2.mr))

print()
for k, v in res.items():
    print(f"  {k:22} mAP50={v['map50']:.4f}  mAP50-95={v['map']:.4f}  P={v['p']:.3f}  R={v['r']:.3f}")

cross = res.get('ai4shipwrecks_only', {}).get('map50')
if cross is not None:
    print(f"\n  cross-dataset {cross:.4f} vs CPU run 0.349 -> "
          f"{'BETTER, worth deploying' if cross > 0.349 else 'no improvement, keep the CPU model'}")


## 8. Export for the backend

Writes `yolov8n-sonar.onnx` (filename kept so `onnx_detector.py` finds it) and a
`metrics.json` containing only figures this notebook measured. The API reads accuracy solely
from that file, so a number can never reach the dashboard unless a real evaluation produced it.


In [ ]:
from datetime import datetime, timezone
import json, shutil
from pathlib import Path
from ultralytics import YOLO

onnx = YOLO(BEST).export(format='onnx', imgsz=640, simplify=True, opset=12)
out = Path('/kaggle/working/models'); out.mkdir(exist_ok=True)
shutil.copy2(onnx, out/'yolov8n-sonar.onnx')   # name the backend expects

c = res['combined']
metrics = {
    'model_version': '2.0.0-combined-colab',
    'map50': round(c['map50'], 4), 'map50_95': round(c['map'], 4),
    'precision': round(c['p'], 4), 'recall': round(c['r'], 4),
    'f1': round(2*c['p']*c['r']/max(c['p']+c['r'], 1e-9), 4),
    'cross_dataset_ai4shipwrecks': res.get('ai4shipwrecks_only'),
    'validated_on': 'SCTD + AI4Shipwrecks held-out split (split by survey line)',
    'trained_on': 'SCTD 1.0 + AI4Shipwrecks tiles, YOLOv8s @ 640px',
    'imgsz': 640,
    'evaluated_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
}
(out/'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))



# /kaggle/working is downloadable from the Output panel once the session
# ends, so write the two files there rather than pushing a browser download.
shutil.copy2(out/'yolov8n-sonar.onnx', WORK/'yolov8n-sonar.onnx')
shutil.copy2(out/'metrics.json', WORK/'metrics.json')
print(f'\nWritten to {WORK} — download both from the Output panel on the right.')


---
## Getting the model out

When the run finishes, open the **Output** panel on the right and download:

```
yolov8n-sonar.onnx
metrics.json
```

Drop both into `backend/models/`, replacing what is there, then commit and push. Render
redeploys automatically. Confirm with:

```bash
curl https://wreckognise-api.onrender.com/api/health
```

`detection_engine` should read `yolov8-onnx`, `trained_weights_loaded` should be `true`.

### Before you ship it

| | SCTD | Cross-dataset |
| --- | --- | --- |
| Deployed now | 0.839 | 0.203 |
| CPU combined run | 0.774 | 0.349 |
| This run | ? | ? |

Training on two corpora usually trades in-distribution accuracy for generalisation, so the
SCTD number may fall while cross-dataset rises. Decide which you want to stand behind, and
say which one you are quoting when you present.
